In [ ]:
# =========================
# 1. INSTALL DEPENDENCIES
# =========================
!pip install -q transformers datasets peft accelerate bitsandbytes trl

# =========================
# 2. IMPORTS
# =========================
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer

# =========================
# 3. CHECK GPU
# =========================
assert torch.cuda.is_available(), "❌ Please enable GPU in Colab!"

# =========================
# 4. LOAD DATASET (IMDB → Instruction Format)
# =========================
dataset = load_dataset("imdb", split="train[:2000]")

def format_example(example):
    label = "positive" if example["label"] == 1 else "negative"
    return {
        "text": f"Review: {example['text']}\nSentiment: {label}"
    }

dataset = dataset.map(format_example)

# =========================
# 5. TOKENIZER
# =========================
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# 6. QLoRA CONFIG (4-bit)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# =========================
# 7. LOAD MODEL
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# =========================
# 8. LoRA CONFIG
# =========================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# =========================
# 9. TRAINER
# =========================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=256,
    tokenizer=tokenizer,
    args=dict(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        fp16=True,
        output_dir="./qlora-output",
        report_to="none"
    )
)

# =========================
# 10. TRAIN
# =========================
trainer.train()

# =========================
# 11. TEST INFERENCE
# =========================
prompt = "Review: This movie was absolutely terrible.\nSentiment:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=10
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))